In [1]:
import pandas as pd
import numpy as np

# Load latest 6 months
files = [
    "data/CRMLSSold202512.csv",
    "data/CRMLSSold202601.csv",
    "data/CRMLSSold202602.csv",
    "data/CRMLSSold202603.csv",
    "data/CRMLSSold202604.csv",
    "data/CRMLSSold202605.csv",
]

df = pd.concat(
    [pd.read_csv(file, low_memory=False) for file in files],
    ignore_index=True
)

print(df.shape)

(112697, 80)


In [2]:
# Keep only Residential Single Family homes
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

print(df.shape)

(55582, 80)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
# Features we will use for preprocessing
features = [
    "CloseDate",
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "City",
    "CountyOrParish",
    "YearBuilt",
    "DaysOnMarket"
]

prep_df = df[features].copy()
prep_df.head()

,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,City,CountyOrParish,YearBuilt,DaysOnMarket
0,2025-12-31,1998000.0,2045.0,4.0,2.0,10080.0,Walnut Creek,Contra Costa,1968.0,0
2,2025-12-31,2214421.0,3050.0,4.0,4.0,34745.0,Woodland Hills,Los Angeles,1957.0,0
3,2025-12-31,1200000.0,1594.0,4.0,2.0,6600.0,San Jose,Santa Clara,1978.0,0
7,2025-12-31,3100000.0,2700.0,5.0,3.0,8262.0,San Jose,Santa Clara,2025.0,0
9,2025-12-31,2900000.0,2948.0,5.0,4.0,9222.0,San Jose,Santa Clara,2023.0,0


In [5]:
# Convert date column
prep_df["CloseDate"] = pd.to_datetime(prep_df["CloseDate"], errors="coerce")

# Create month column for train/test split
prep_df["CloseMonth"] = prep_df["CloseDate"].dt.to_period("M").astype(str)

prep_df[["CloseDate", "CloseMonth"]].head()

,CloseDate,CloseMonth
0,2025-12-31,2025-12
2,2025-12-31,2025-12
3,2025-12-31,2025-12
7,2025-12-31,2025-12
9,2025-12-31,2025-12


In [6]:
# Check missing values
prep_df.isna().sum()

CloseDate                   0
ClosePrice                  0
LivingArea                 30
BedroomsTotal               0
BathroomsTotalInteger       1
LotSizeSquareFeet        1033
City                       11
CountyOrParish              0
YearBuilt                  34
DaysOnMarket                0
CloseMonth                  0
dtype: int64

In [7]:
# Drop rows missing target or date
prep_df = prep_df.dropna(subset=["ClosePrice", "CloseDate"])

# Impute numerical missing values using median
num_cols = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "DaysOnMarket"
]

for col in num_cols:
    prep_df[col] = prep_df[col].fillna(prep_df[col].median())

# Impute categorical missing values
cat_cols = ["City", "CountyOrParish"]

for col in cat_cols:
    prep_df[col] = prep_df[col].fillna("Unknown")

In [8]:
# Remove obvious invalid values
prep_df = prep_df[
    (prep_df["ClosePrice"] > 0) &
    (prep_df["LivingArea"] > 0) &
    (prep_df["BedroomsTotal"] > 0) &
    (prep_df["BathroomsTotalInteger"] > 0) &
    (prep_df["LotSizeSquareFeet"] > 0)
].copy()

prep_df.shape

(55488, 11)

In [9]:
# One-hot encode categorical variables
prep_encoded = pd.get_dummies(
    prep_df,
    columns=cat_cols,
    drop_first=True
)

prep_encoded.head()

,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,DaysOnMarket,CloseMonth,City_Acampo,...,CountyOrParish_Sonoma,CountyOrParish_Stanislaus,CountyOrParish_Sutter,CountyOrParish_Tehama,CountyOrParish_Trinity,CountyOrParish_Tulare,CountyOrParish_Tuolumne,CountyOrParish_Ventura,CountyOrParish_Yolo,CountyOrParish_Yuba
0,2025-12-31,1998000.0,2045.0,4.0,2.0,10080.0,1968.0,0,2025-12,False,...,False,False,False,False,False,False,False,False,False,False
2,2025-12-31,2214421.0,3050.0,4.0,4.0,34745.0,1957.0,0,2025-12,False,...,False,False,False,False,False,False,False,False,False,False
3,2025-12-31,1200000.0,1594.0,4.0,2.0,6600.0,1978.0,0,2025-12,False,...,False,False,False,False,False,False,False,False,False,False
7,2025-12-31,3100000.0,2700.0,5.0,3.0,8262.0,2025.0,0,2025-12,False,...,False,False,False,False,False,False,False,False,False,False
9,2025-12-31,2900000.0,2948.0,5.0,4.0,9222.0,2023.0,0,2025-12,False,...,False,False,False,False,False,False,False,False,False,False


In [10]:
# Time-based train/test split
# Most recent month = test set
latest_month = prep_encoded["CloseMonth"].max()

test_df = prep_encoded[prep_encoded["CloseMonth"] == latest_month].copy()
train_df = prep_encoded[prep_encoded["CloseMonth"] != latest_month].copy()

print("Latest month used as test set:", latest_month)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Latest month used as test set: 2026-05
Train shape: (43054, 968)
Test shape: (12434, 968)


In [11]:
# Separate X and y
drop_cols = ["ClosePrice", "CloseDate", "CloseMonth"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["ClosePrice"]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df["ClosePrice"]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(43054, 965) (43054,)
(12434, 965) (12434,)


In [12]:
# Normalize numerical features
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train.head()

,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,DaysOnMarket,City_Acampo,City_Acton,City_Adelanto,City_Agoura Hills,...,CountyOrParish_Sonoma,CountyOrParish_Stanislaus,CountyOrParish_Sutter,CountyOrParish_Tehama,CountyOrParish_Trinity,CountyOrParish_Tulare,CountyOrParish_Tuolumne,CountyOrParish_Ventura,CountyOrParish_Yolo,CountyOrParish_Yuba
0,-0.000387,0.525189,-0.562457,-0.021257,-0.288298,-0.732191,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0.982780,0.525189,1.210441,-0.019938,-0.685521,-0.732191,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,-0.441590,0.525189,-0.562457,-0.021443,0.072814,-0.732191,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7,0.640383,1.562793,0.323992,-0.021354,1.770040,-0.732191,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9,0.882996,1.562793,1.210441,-0.021303,1.697817,-0.732191,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [13]:
# Save cleaned dataset
prep_encoded.to_csv("data/cleaned_sold_single_family.csv", index=False)